# MNIST MLP3 baseline comparison — zoomed diagnostics

This notebook validates and compares the persisted three-seed **SGD + Nesterov**, **AdamW**, and **Muon + auxiliary AdamW** controls. It reads only from the versioned suite directory `mnist_mlp3_recipe_v3`; old unversioned 20-epoch artifacts are ignored.

All uncertainty intervals use complete training runs as the unit of replication. Validation loss selects the reported best checkpoint; the official MNIST test set is monitoring-only.

It retains the full-range comparison plots and adds post-transient zoomed copies of the same plots. The WeightWatcher-alpha comparison also includes a second late-training view centered on the neighborhood of `alpha = 2`. All displayed DataFrames show every row and every column.


In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir():
        ROOT = candidate
        break
    if (path / 'rg_baselines').is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError('Run from a current clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rg_baselines import (
    MNIST_REFERENCE_INITIALIZATION,
    MNIST_REFERENCE_RECIPE_VERSION,
    MNIST_REFERENCE_SUITE_SLUG,
)
from rg_baselines.comparison import (
    LAYER_ORDER,
    OPTIMIZER_COLORS,
    OPTIMIZER_LABELS,
    OPTIMIZER_ORDER,
    run_baseline_comparison,
)
from rg_baselines.comparison_plotting import (
    PERFORMANCE_PLOTS,
    SPECTRAL_PLOTS,
    plot_performance_metric,
    plot_spectral_metric,
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

BASE_RUN_ROOT = Path(
    os.environ.get('RG_BASELINE_RUN_ROOT', ROOT / 'runs')
).expanduser().resolve()
RUN_ROOT = BASE_RUN_ROOT / MNIST_REFERENCE_SUITE_SLUG
OUTPUT_DIR = RUN_ROOT / 'comparison'

legacy_directories = [
    BASE_RUN_ROOT / optimizer
    for optimizer in OPTIMIZER_ORDER
    if (BASE_RUN_ROOT / optimizer).is_dir()
]
if legacy_directories:
    print('Ignoring legacy unversioned result directories:')
    for directory in legacy_directories:
        print('  -', directory)

required_manifests = [
    RUN_ROOT / optimizer / 'replicate_manifest.json'
    for optimizer in OPTIMIZER_ORDER
]
missing = [path for path in required_manifests if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'The current recipe-v3 baseline suite is incomplete. Run these '
        'three source notebooks first, in order:\n'
        '  1. MNIST_MLP3_SGD_Momentum_Baseline.ipynb\n'
        '  2. MNIST_MLP3_AdamW_Baseline.ipynb\n'
        '  3. MNIST_MLP3_SGD_Momentum_Muon_Baseline.ipynb\n'
        'Missing manifests:\n' + '\n'.join(f'  - {path}' for path in missing)
    )

print('MNIST reference recipe:', MNIST_REFERENCE_RECIPE_VERSION)
print('versioned suite root:', RUN_ROOT)


In [ ]:
result = run_baseline_comparison(
    RUN_ROOT,
    output_dir=OUTPUT_DIR,
    show_plots=True,
)
assert len(result.seeds) == 3
assert result.epochs == 30
for optimizer, manifest in result.manifests.items():
    config = manifest['config_template']
    assert config['recipe_version'] == MNIST_REFERENCE_RECIPE_VERSION
    assert config['initialization'] == MNIST_REFERENCE_INITIALIZATION
print('comparison outputs:', OUTPUT_DIR)


## Post-transient zoomed comparison plots

The original full-range plots above remain unchanged. The additional plot suite below starts at epoch 3, omitting initialization and the first two training epochs so the later variation is visible. The two alpha figures use explicitly bounded axes; confidence bands and seed trajectories outside a displayed bound are intentionally clipped in the zoomed view, while remaining visible in the original full-range figure.


In [ ]:
ZOOM_START_EPOCH = 3
ALPHA_DETAIL_START_EPOCH = 10
ZOOM_PLOT_DIR = OUTPUT_DIR / 'plots' / f'zoom_epoch_{ZOOM_START_EPOCH}_onward'
ZOOM_PLOT_DIR.mkdir(parents=True, exist_ok=True)

performance_zoom = result.performance.loc[
    result.performance['epoch'].astype(int).ge(ZOOM_START_EPOCH)
].copy()
performance_summary_zoom = result.performance_summary.loc[
    result.performance_summary['epoch'].astype(int).ge(ZOOM_START_EPOCH)
].copy()
spectral_zoom = result.spectral_metrics.loc[
    result.spectral_metrics['epoch'].astype(int).ge(ZOOM_START_EPOCH)
].copy()
spectral_summary_zoom = result.spectral_summary.loc[
    result.spectral_summary['epoch'].astype(int).ge(ZOOM_START_EPOCH)
].copy()

zoom_plot_paths = []

for metric, ylabel, filename, reference, percent in PERFORMANCE_PLOTS:
    if metric not in performance_zoom.columns:
        continue
    if not performance_zoom[metric].notna().any():
        continue
    zoom_plot_paths.append(
        plot_performance_metric(
            performance_zoom,
            performance_summary_zoom,
            metric=metric,
            ylabel=ylabel,
            path=ZOOM_PLOT_DIR / filename,
            seeds=result.seeds,
            optimizer_order=OPTIMIZER_ORDER,
            labels=OPTIMIZER_LABELS,
            colors=OPTIMIZER_COLORS,
            horizontal_reference=reference,
            percent=percent,
            show=True,
        )
    )

for metric, ylabel, filename, reference in SPECTRAL_PLOTS:
    if metric == 'alpha':
        continue
    if metric not in spectral_zoom.columns:
        continue
    zoom_plot_paths.append(
        plot_spectral_metric(
            spectral_zoom,
            spectral_summary_zoom,
            metric=metric,
            ylabel=ylabel,
            path=ZOOM_PLOT_DIR / filename,
            seeds=result.seeds,
            optimizer_order=OPTIMIZER_ORDER,
            layer_order=LAYER_ORDER,
            labels=OPTIMIZER_LABELS,
            colors=OPTIMIZER_COLORS,
            horizontal_reference=reference,
            show=True,
        )
    )


def plot_bounded_alpha(
    *,
    start_epoch,
    y_limits_by_layer,
    title,
    filename,
):
    figure, axes = plt.subplots(
        1,
        len(LAYER_ORDER),
        figsize=(16.0, 4.8),
        sharex=True,
    )
    if len(LAYER_ORDER) == 1:
        axes = [axes]

    for axis, layer in zip(axes, LAYER_ORDER, strict=True):
        for optimizer in OPTIMIZER_ORDER:
            raw = result.spectral_metrics.loc[
                result.spectral_metrics['optimizer'].eq(optimizer)
                & result.spectral_metrics['layer'].astype(str).eq(str(layer))
                & result.spectral_metrics['epoch'].astype(int).ge(start_epoch)
            ]
            for seed in result.seeds:
                rows = raw.loc[
                    raw['seed'].astype(int).eq(int(seed))
                ].sort_values('epoch')
                axis.plot(
                    rows['epoch'],
                    rows['alpha'],
                    color=OPTIMIZER_COLORS[optimizer],
                    linewidth=0.8,
                    alpha=0.15,
                )

            rows = result.spectral_summary.loc[
                result.spectral_summary['optimizer'].eq(optimizer)
                & result.spectral_summary['layer'].astype(str).eq(str(layer))
                & result.spectral_summary['metric'].eq('alpha')
                & result.spectral_summary['epoch'].astype(int).ge(start_epoch)
            ].sort_values('epoch')
            if rows.empty:
                continue
            x = rows['epoch'].astype(float).to_numpy()
            mean = rows['mean'].astype(float).to_numpy()
            low = rows['ci_low'].astype(float).to_numpy()
            high = rows['ci_high'].astype(float).to_numpy()
            axis.plot(
                x,
                mean,
                color=OPTIMIZER_COLORS[optimizer],
                linewidth=2.2,
                label=OPTIMIZER_LABELS[optimizer],
            )
            axis.fill_between(
                x,
                low,
                high,
                color=OPTIMIZER_COLORS[optimizer],
                alpha=0.12,
            )

        axis.axhline(
            2.0,
            color='black',
            linewidth=1.0,
            linestyle='--',
            alpha=0.7,
            label=r'$\alpha = 2$',
        )
        axis.set_xlim(start_epoch, result.epochs)
        axis.set_ylim(*y_limits_by_layer[layer])
        axis.set(title=str(layer).upper(), xlabel='Epoch')
        axis.grid(alpha=0.2)

    axes[0].set_ylabel('WeightWatcher alpha')
    axes[-1].legend(frameon=False, loc='best')
    figure.suptitle(title, y=1.02)
    figure.tight_layout()
    path = ZOOM_PLOT_DIR / filename
    figure.savefig(path, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(figure)
    return path


post_transient_alpha_path = plot_bounded_alpha(
    start_epoch=ZOOM_START_EPOCH,
    y_limits_by_layer={
        'fc1': (1.5, 10.0),
        'fc2': (1.5, 10.0),
        'fc3': (2.0, 7.5),
    },
    title='WeightWatcher alpha after the initial transient',
    filename='20_weightwatcher_alpha_95ci_bounded.png',
)
zoom_plot_paths.append(post_transient_alpha_path)

alpha_near_two_path = plot_bounded_alpha(
    start_epoch=ALPHA_DETAIL_START_EPOCH,
    y_limits_by_layer={
        'fc1': (1.5, 4.5),
        'fc2': (1.5, 6.0),
        'fc3': (1.5, 5.0),
    },
    title=r'Late-training WeightWatcher alpha near $\alpha = 2$',
    filename='20_weightwatcher_alpha_95ci_near_two.png',
)
zoom_plot_paths.append(alpha_near_two_path)

print('zoomed comparison plots:', ZOOM_PLOT_DIR)


## Final and validation-selected performance

The validation-selected table evaluates the protected test set only after the checkpoint epoch has been chosen from validation loss.


In [ ]:
display(
    result.terminal_by_seed.sort_values(
        ['checkpoint', 'optimizer_label', 'seed']
    )
)
display(
    result.terminal_summary.sort_values(
        ['metric', 'checkpoint', 'optimizer_label']
    )
)
assert result.terminal_summary['n'].eq(3).all()


## Validation convergence and matched-seed contrasts

Convergence thresholds are defined on validation accuracy. Paired differences are reported for both final and validation-selected checkpoints.


In [ ]:
display(result.convergence_by_seed.sort_values(['optimizer_label', 'seed']))
display(result.convergence_summary.sort_values(['metric', 'optimizer_label']))
display(
    result.paired_final_differences.sort_values(
        ['checkpoint', 'metric', 'contrast']
    )
)


## Layerwise WeightWatcher summary

`alpha`, `ERG_gap`, and randomized-MP `num_traps` are direct WeightWatcher outputs. Each layer is summarized across the three complete runs; layers are not treated as additional replicates.


In [ ]:
spectral = result.spectral_summary[
    result.spectral_summary['metric'].isin([
        'alpha', 'ERG_gap', 'num_traps',
        'm_midpoint', 'trace_log_midpoint_per_eval',
    ])
].sort_values(['metric', 'optimizer_label', 'layer', 'epoch'])
assert spectral['n'].eq(3).all()
display(spectral)


The historical result-directory key `sgd_momentum_muon` is retained only for compatibility. Its implementation and label are **Muon + auxiliary AdamW**. No test metric is used for learning-rate choice, checkpoint selection, or stopping.
